![](https://drive.google.com/uc?export=view&id=1-5X9OUkA-C2Ih1gOS9Jd7GmkTWUEpDg1)

**Asignatura:** *Deep Learning*

**Profesor:** Juan Bekios Calfa

**Fecha de entrega:** 22 de abril de 2026
---

#Laboratorio 01: Clasificación de Deterioro Cognitivo Usando Naive Bayes con Leave-One-Out

##1. Introducción

El deterioro cognitivo es una condición progresiva que afecta la memoria y otras funciones mentales, representando un gran desafío para el diagnóstico clínico temprano. El uso de pruebas psicométricas como el Addenbrooke's Cognitive Examination III (ACE-III), junto con algoritmos de aprendizaje automático, ha demostrado ser prometedor para asistir en la detección automatizada de demencia.

Este laboratorio se enfoca en el uso de un subconjunto de 15 y 24 ítems del ACE-III, con entradas binarias (acierto/error), para predecir el nivel de deterioro según la escala Global Deterioration Scale (GDS), usando una implementación de Naive Bayes sin librerías externas.

##2. Objetivos

1. Entender la estructura de los datos derivados del ACE-III.

2. Aplicar técnicas básicas de selección/extracción de características.

3. Implementar el algoritmo Naive Bayes manualmente.

4. Evaluar el rendimiento del modelo con validación Leave-One-Out (LOOCV).

5. Comparar el desempeño con datasets de 15 y 24 atributos.

6. Analizar y discutir resultados relevantes para el diagnóstico clínico.



## 3. Comprensión de los Datos

###3.1 Descripción General

El conjunto de datos utilizado proviene de una versión reducida del ACE-III, compuesta por 15 o 24 atributos binarios que representan aciertos o errores en ítems de orientación y memoria. Cada fila representa a una persona mayor evaluada, y la etiqueta de clase corresponde a un valor categórico derivado de la Escala de Deterioro Global (GDS), la cual evalúa siete niveles progresivos de deterioro cognitivo.

###3.2 Codificaciones de la Variable de Clase (GDS)

Para facilitar el proceso de clasificación y mitigar los efectos del desbalance de clases, el estudio original propuso distintas agrupaciones (codificaciones) de las 7 categorías originales de la GDS. Estas codificaciones se utilizan para transformar el problema en tareas de clasificación con 2, 3 o 6 clases:

#### Tabla 1. Agrupaciones de categorías GDS utilizadas para entrenamiento y prueba

| Codificación | Agrupación de categorías GDS             | Nº de Clases | Descripción                                               |
|--------------|-------------------------------------------|--------------|-----------------------------------------------------------|
| GDS          | 1, 2, 3, 4, 5, 6–7                        | 6            | Se agrupan solo las categorías más severas (6 y 7)        |
| GDS_1        | 1–3, 4–5, 6–7                             | 3            | Agrupa en leve, moderado y severo                         |
| GDS_2        | 1–2, 3, 4–7                               | 3            | Distingue deterioro leve como clase independiente         |
| GDS_3        | 1–3, 4–7                                  | 2            | Binaria: sin o leve deterioro vs. moderado/severo         |
| GDS_4        | 1, 2–4, 5–7                               | 3            | Mantiene categoría 1 separada                             |
| GDS_5        | 1, 2–3, 4–7                               | 3            | Categoriza deterioro leve como clase intermedia aislada   |


Esto permite realizar múltiples experimentos y comparar cómo varía el rendimiento del modelo según la granularidad de la clasificación.

###3.3 Distribución de Clases
El conjunto de datos presenta un fuerte desbalance, donde la mayoría de los casos se concentran en las categorías GDS 2 y 3 (deterioro muy leve y leve), y muy pocos en GDS 6 y 7 (severo y muy severo). Esto se debe considerar en la interpretación de resultados y elección de técnicas de validación.

### 3.4 Cargar los datos

In [ ]:
# Cargar los archivos desde una fuente de datos (Google Drive)
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive/MyDrive/COQ-ESC-UCN/Projects/2020/Propuestos/Psicologia-DeterioroCognitivo/Data

In [ ]:
# Importar librerías que no están instaladas en el paquete de Python
!pip install pyreadstat

In [ ]:
# Cargar archivo con 14 atributos
import pandas as pd #Librerías para manejar dataframe

filename = '15 atributos R0-R5.sav'
df_15 = pd.read_spss(filename)
df_15.head()

####3.4.1 Analizar los datos

Contamos las cantidad de valores clase para las agrupaciones GDS.

In [ ]:
frecuencia_valor_clase = df_15['GDS'].value_counts(normalize=False)
print(frecuencia_valor_clase.sort_index(ascending=True))

Probamos con la agrupación GDS_R1

In [ ]:
frecuencia_valor_clase = df_15['GDS_R1'].value_counts(normalize=False)
print(frecuencia_valor_clase.sort_index(ascending=True))

Se puede implementar un gráfico para ver la distribución de los datos para las distintas agrupaciones.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Lista de columnas codificadas
experiments = ['GDS', 'GDS_R1', 'GDS_R2', 'GDS_R3', 'GDS_R4', 'GDS_R5']
width = 0.15
sum_width = 0

plt.figure(figsize=(8, 6))

for exp in experiments:
    print(exp)

    # Obtener conteo de valores únicos ordenados por índice
    counts = df_15[exp].value_counts().sort_index()
    print(counts)

    # Crear lista completa de clases (del 1 al 7), completando con 0 si falta alguna
    classes = list(range(1, 8))
    data = [counts.get(c, 0) for c in classes]

    # Graficar
    plt.bar(np.array(classes) + sum_width, data, width=width, label=exp)

    sum_width += width

    # Mostrar conteo en consola
    print(counts)
    print('=' * 60)

# Etiquetas y leyenda
plt.title('Frecuencias de valores de clases vs tipos de codificación')
plt.xlabel('Etiqueta de la clase (GDS)')
plt.ylabel('Frecuencia')
plt.xticks(np.arange(1, 8), labels=[str(i) for i in range(1, 8)])
plt.legend()
plt.show()

## 4. Preparación de los Datos

###4.1 Verificación de binariedad y valores nulos

In [ ]:
# Lista de columnas a excluir (Revisar error)
columnas_excluir = ['GDS', 'GDS_R1', 'GDS_R2', 'GDS_R3', 'GDS_R4', 'GDS_R5', 'ID']

# Seleccionar solo las columnas que NO están en la lista de exclusión
df_binarias = df_15.drop(columns=columnas_excluir, errors='ignore')

# Verificación de binariedad
assert df_binarias.isin([0, 1]).all().all(), "Hay columnas con valores no binarios"

###4.2 Selección de características

La selección de variables permite identificar las características más relevantes para mejorar el rendimiento, simplicidad y generalización de un modelo. Al reducir atributos irrelevantes o redundantes, se evita el sobreajuste y se acelera el procesamiento. Existen tres enfoques principales:

**Filtros**: usan métricas estadísticas independientes (como varianza o chi-cuadrado).

***Wrappers***: evalúan subconjuntos usando el rendimiento del modelo.

**Embebidos**: integran la selección dentro del proceso de entrenamiento.

####4.2.1 Selección de características por varianza

El filtro por varianza elimina las variables que presentan poca variabilidad entre los datos, ya que se considera que aportan poca información al modelo. Si una característica tiene la misma o casi la misma cantidad en todos los casos (por ejemplo, casi todos son ceros), es probable que no sea útil para la predicción.

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# Quitar columnas GDS y etiquetas, quedarnos solo con variables binarias
X = df_15.drop(columns=['ID', 'GDS', 'GDS_R1', 'GDS_R2', 'GDS_R3', 'GDS_R4', 'GDS_R5'], errors='ignore')
y = df_15['GDS']

# Filtro de varianza (remueve columnas con varianza menor a 0.02)
selector = VarianceThreshold(threshold=0.02)
X_var = selector.fit_transform(X)

# Obtener nombres de columnas seleccionadas
selected_cols_var = X.columns[selector.get_support()]
print("Características seleccionadas por varianza:")
print(list(selected_cols_var))

####4.2.2 Selección de características utilizando $\chi^2$ (Chi2)

El filtro chi-cuadrado evalúa la independencia entre cada característica y la variable objetivo. Selecciona las variables que tienen mayor dependencia estadística con la clase, es decir, aquellas cuyo valor está relacionado con la categoría a predecir. **Es útil para datos categóricos o binarios**.

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2
import pandas as pd

# Definir X e y
X = df_15.drop(columns=['ID', 'GDS', 'GDS_R1', 'GDS_R2', 'GDS_R3', 'GDS_R4', 'GDS_R5'], errors='ignore')
y = df_15['GDS']

# Aplicar chi2
selector = SelectKBest(score_func=chi2, k=15)
selector.fit_transform(X, y)

# Características seleccionadas
selected_cols_chi2 = X.columns[selector.get_support()]
print("Características seleccionadas por chi2:")
print(list(selected_cols_chi2), end='\n'*2)

# Crear un DataFrame con las puntuaciones de chi2 y los nombres de las variables
chi2_scores = pd.DataFrame({
    'Feature': X.columns[selector.get_support()],
    'Chi2 Score': selector.scores_[selector.get_support()]
}).sort_values(by='Chi2 Score', ascending=False)

# Mostrar resultados ordenados
print("Características seleccionadas por chi2 (ordenadas por importancia):")
print(chi2_scores)

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2
import pandas as pd

# X: atributos (binarios), y: variable objetivo
X = df_15.drop(columns=['ID','GDS', 'GDS_R1', 'GDS_R2', 'GDS_R3', 'GDS_R4', 'GDS_R5'], errors='ignore')
y = df_15['GDS']  # o 'GDS_R1', etc.

# Aplicar chi²
selector = SelectKBest(score_func=chi2, k='all')
selector.fit(X, y)

# Crear DataFrame con resultados
chi2_scores = pd.DataFrame({
    'Feature': X.columns,
    'Chi2 Score': selector.scores_,
    'p-value': selector.pvalues_
}).sort_values(by='Chi2 Score', ascending=False)

print(chi2_scores)

####4.2.4 Método Wrapper: RFE (*Recursive Feature Elimination*)

Usamos un clasificador (por ejemplo, `LogisticRegression`) para evaluar y eliminar variables iterativamente.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

X = df_15.drop(columns=['ID','GDS', 'GDS_R1', 'GDS_R2', 'GDS_R3', 'GDS_R4', 'GDS_R5'], errors='ignore')
y = df_15['GDS']

# Modelo base
model = LogisticRegression(max_iter=1000)

# Seleccionar las 10 mejores características con RFE
selector = RFE(estimator=model, n_features_to_select=10)
selector = selector.fit(X, y)

selected_cols_rfe = X.columns[selector.get_support()]
print("Características seleccionadas por RFE:")
print(list(selected_cols_rfe))

Ordenar por aquellas que tienen más importancia

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pandas as pd

# Definir X e y
X = df_15.drop(columns=['GDS', 'GDS_R1', 'GDS_R2', 'GDS_R3', 'GDS_R4', 'GDS_R5'], errors='ignore')
y = df_15['GDS']

# Modelo base
model = LogisticRegression(max_iter=10000)

# Selección con RFE
selector = RFE(estimator=model, n_features_to_select=10)
selector = selector.fit(X, y)

# Crear DataFrame con nombres y ranking
rfe_ranking = pd.DataFrame({
    'Feature': X.columns,
    'Ranking': selector.ranking_,
    'Selected': selector.support_
})

# Filtrar solo las seleccionadas y ordenarlas por ranking (más importantes = ranking 1)
top_features = rfe_ranking[rfe_ranking['Selected']].sort_values(by='Ranking')

print("Características seleccionadas por RFE (ordenadas por importancia):")
print(top_features)

Problemas de convergencia en la versión anterior. Se prueba con otro modelo.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
import pandas as pd

# Definir X e y
X = df_15.drop(columns=['ID', 'GDS', 'GDS_R1', 'GDS_R2', 'GDS_R3', 'GDS_R4', 'GDS_R5'], errors='ignore')
y = df_15['GDS']

# Modelo base: árbol de decisión
model = DecisionTreeClassifier(random_state=42)

# Seleccionar las 10 mejores características con RFE
selector = RFE(estimator=model, n_features_to_select=10)
selector = selector.fit(X, y)

# Crear DataFrame con ranking
rfe_ranking = pd.DataFrame({
    'Feature': X.columns,
    'Ranking': selector.ranking_,
    'Selected': selector.support_
})

# Mostrar top features
top_features = rfe_ranking[rfe_ranking['Selected']].sort_values(by='Ranking')

print("Características seleccionadas por RFE con Árbol de Decisión:")
print(top_features)

# Obtener nombres de las variables seleccionadas
selected_feature_names = X.columns[selector.support_]

# Crear nuevo DataFrame solo con las top features
df_top = df_15[selected_feature_names].copy()

# Agregar la variable objetivo con el nombre que espera leave_one_out
df_top["label"] = y.values

print("\nDataFrame final para leave_one_out:")
print(df_top.head())  # UTILIZAREMOS ESTAS FEATURES PARA ENTRENAR EL MODELO

###Forward elimination (Otra alternativa)

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# Definir X e y
X = df_15.drop(columns=['ID', 'GDS', 'GDS_R1', 'GDS_R2', 'GDS_R3', 'GDS_R4', 'GDS_R5'], errors='ignore')
y = df_15['GDS']

# Modelo base
modelo = KNeighborsClassifier(n_neighbors=3)

# Selector forward
sfs = SequentialFeatureSelector(
    estimator=modelo,
    n_features_to_select=5,     # número de variables a seleccionar
    direction="forward",        # selección hacia adelante
    scoring="accuracy",         # métrica
    cv=5,                       # validación cruzada
    n_jobs=-1
)

# Pipeline opcional: escalado + selección
pipe = Pipeline([
    ("scaler", MinMaxScaler()),
    ("selector", sfs)
])

# Ajustar
pipe.fit(X, y)

# Obtener variables seleccionadas
selected_cols = X.columns[pipe.named_steps["selector"].get_support()]

print("Características seleccionadas por forward selection:")
print(list(selected_cols))

####4.2.5 Extracción de características con PCA

PCA es una técnica de reducción de dimensionalidad que transforma las variables originales en nuevas variables no correlacionadas llamadas componentes principales, que explican la mayor parte de la varianza de los datos.

* PCA no selecciona variables originales, sino que crea nuevas combinaciones lineales.

* Es útil para eliminar redundancia entre atributos.

* Aunque tus datos son binarios, PCA puede funcionar bien si hay correlaciones entre variables.



In [ ]:
from sklearn.decomposition import PCA
import pandas as pd
import matplotlib.pyplot as plt

# 1. Preparamos X quitando columnas no deseadas
X = df_15.drop(columns=['ID', 'GDS', 'GDS_R1', 'GDS_R2', 'GDS_R3', 'GDS_R4', 'GDS_R5'], errors='ignore')

# 2. Aplicar PCA (sin escalado porque los datos ya son binarios)
pca = PCA(n_components=None)  # Mantener todos los componentes
X_pca = pca.fit_transform(X)

# 3. Visualizar la varianza explicada
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(pca.explained_variance_ratio_) + 1),
         pca.explained_variance_ratio_.cumsum(), marker='o')
plt.xlabel('Número de Componentes Principales')
plt.ylabel('Varianza Acumulada Explicada')
plt.title('PCA - Varianza explicada acumulada')
plt.grid()

Elegir un número de componentes que expliquen, por ejemplo, el 95% de la varianza:

Para elegir el número de componentes que expliquen el 95% de la varianza, se calcula la suma acumulada de los **autovalores normalizados** del análisis de componentes principales:

$$
\frac{\sum_{i=1}^{k} \lambda_i}{\sum_{i=1}^{n} \lambda_i} \geq 0.95
$$

donde  $\lambda_i$ es la varianza explicada por el componente $i$, y $k$ es el menor número de componentes que cumple la condición.

In [ ]:
pca_95 = PCA(n_components=0.95)
X_pca_95 = pca_95.fit_transform(X)
print(f"Número de componentes que explican el 95% de la varianza: {pca_95.n_components_}")

Los resultados se pueden guardar en un Dataframe:

In [ ]:
X_pca_df = pd.DataFrame(X_pca_95, columns=[f'PC{i+1}' for i in range(pca_95.n_components_)])
X_pca_df.head()

## 5. Modelamiento


### 📘 Clasificador Naive Bayes (modelo multiclase binario)

El clasificador se basa en el **Teorema de Bayes**:

$$
P(C_k \mid \mathbf{x}) = \frac{P(C_k) \cdot \prod_{i=1}^{n} P(x_i \mid C_k)}{P(\mathbf{x})}
$$

Donde:
- $\mathbf{x} = (x_1, x_2, ..., x_n) $ son los atributos binarios de entrada,
- $C_k $ es una clase posible (por ejemplo, un nivel de GDS),
- $ P(C_k) $ es la probabilidad a priori de la clase $ C_k $,
- $ P(x_i \mid C_k) $ es la probabilidad condicional del valor del atributo dado la clase.


### Supuestos del modelo:

- **Independencia condicional**: los atributos son independientes entre sí dado la clase.
- **Distribución discreta**: las probabilidades se estiman con frecuencias relativas (y suavizado de Laplace para evitar ceros).



### Clasificación:

Se predice la clase $ C_k $ que **maximiza la probabilidad posterior**:

$$
\hat{C} = \arg\max_{C_k} \left( \log P(C_k) + \sum_{i=1}^{n} \log P(x_i \mid C_k) \right)
$$

Esto es lo que implementas con logaritmos para evitar problemas numéricos con productos muy pequeños.

In [ ]:
from collections import defaultdict
import math

class NaiveBayes:
    def fit(self, X, y):
        self.classes = set(y)
        self.class_counts = defaultdict(int)
        self.feature_counts = defaultdict(lambda: defaultdict(int))
        self.total = len(y)

        for xi, label in zip(X, y):
            self.class_counts[label] += 1
            for i, val in enumerate(xi):
                self.feature_counts[label][(i, val)] += 1

    def predict_one(self, xi):
        scores = {}
        for c in self.classes:
            logp = math.log(self.class_counts[c] / self.total)
            for i, val in enumerate(xi):
                freq = self.feature_counts[c][(i, val)] + 1
                total = self.class_counts[c] + 2
                logp += math.log(freq / total)
            scores[c] = logp
        return max(scores, key=scores.get)

##6. Validación *Leave-One-Out*

In [ ]:
from sklearn.metrics import classification_report

def leave_one_out(df):
    X = df.drop("label", axis=1).values.tolist()
    y = df["label"].tolist()
    y_true, y_pred = [], []

    for i in range(len(X)):
        X_train = X[:i] + X[i+1:]
        y_train = y[:i] + y[i+1:]
        x_test, y_test = X[i], y[i]

        model = NaiveBayes()
        model.fit(X_train, y_train)
        y_pred.append(model.predict_one(x_test))
        y_true.append(y_test)

    print(classification_report(y_true, y_pred, digits=3))

In [ ]:
# Ejecutar leave-one-out
leave_one_out(df_top)

##7. Resultados

Explicar para cada uno del conjunto de etiquetas como fueron realizados los experimentos. Generar dos grupos de experimentos, uno con selección de variables y otro con extracción de variables.

Experimento:


*   Modelo para etiquetas GDS: Los experimentos se realizó ...
*   Modelo para etiquetas GDS_R1:
*   Modelo para etiquetas GDS_R2:
*   Modelo para etiquetas GDS_R3:
*   Modelo para etiquetas GDS_R4:
*   Modelo para etiquetas GDS_R5:


Probar por cada modelo, Y mostrar los resultados con las métricas utilizadas para clasificación.

### Tabla: Resultados por codificación de GDS (métricas promedio macro)

| Codificación   |   accuracy |   precision |   recall |   f1_score |
|:---------------|-----------:|------------:|---------:|-----------:|
| GDS            |      0.721 |       0.713 |    0.705 |      0.707 |
| GDS_R1         |      0.752 |       0.748 |    0.750 |      0.749 |
| GDS_R2         |      0.761 |       0.758 |    0.755 |      0.756 |
| GDS_R3         |      0.803 |       0.800 |    0.798 |      0.799 |
| GDS_R4         |      0.748 |       0.743 |    0.740 |      0.741 |
| GDS_R5         |      0.739 |       0.734 |    0.732 |      0.733 |

Nota: La tabla es solo un ejemplo, los resultados se deben calcular como parte del laboratorio.

Mostrar un gráfico que resuma los resultados.

In [ ]:
# Código ejemplo
import matplotlib.pyplot as plt

# Graficar las métricas
#plt.figure(figsize=(12, 6))
#results_df.plot(kind='bar', figsize=(12, 6), colormap='Set2')

#plt.title('Comparación de Métricas por Codificación de GDS')
#plt.ylabel('Valor')
#plt.xlabel('Codificación')
#plt.ylim(0, 1.05)
#plt.grid(axis='y', linestyle='--', alpha=0.7)
#plt.legend(title='Métrica')
#plt.tight_layout()

##8. Conclusiones

A partir de los resultados escribir en párrafos sus conclusiones. Considere las siguientes preguntas para poder responder adecuadamente las conclusiones.

**Sobre los resultados del modelo**
¿Cuál fue la codificación de GDS que entregó el mejor desempeño general?

¿Qué tan precisos fueron los modelos en predecir clases menos representadas (por ejemplo, deterioro severo)?

¿Se observaron diferencias notables entre accuracy, precision, recall y f1-score? ¿Qué nos dicen esas diferencias?

¿El modelo mostró un buen equilibrio entre clases o favoreció a las más frecuentes?

**Sobre la selección de variables (chi²)**
¿Las variables seleccionadas con chi² parecen tener sentido clínico o neuropsicológico?

¿La selección de atributos mejoró el rendimiento en comparación con usar todos?

¿Qué variables se repiten entre distintas codificaciones de GDS?

**Sobre el modelo Naive Bayes**
¿El modelo Naive Bayes fue suficiente para capturar las relaciones entre los atributos y las clases?

¿Existen indicios de que un modelo más complejo (como Random Forest) podría mejorar los resultados?

¿Fue estable el modelo en todas las codificaciones, o su rendimiento fue inconsistente?

**Sobre el diseño experimental**
¿La validación Leave-One-Out fue adecuada para este problema y tamaño de muestra?

¿Habría sido útil aplicar técnicas de balanceo o normalización?

¿Qué limitaciones tiene este experimento en términos de generalización o aplicabilidad clínica?

**Proyección y utilidad**
¿Pueden estos resultados usarse como apoyo complementario al diagnóstico clínico?

¿Qué próximos pasos se podrían seguir para mejorar el sistema de clasificación?



#ANEXO: Código complementario

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from collections import defaultdict
import math

# Cargar los datos (asegúrate que df_15 está cargado)
df = df_15.copy()

# Codificaciones GDS
gds_columns = ['GDS', 'GDS_R1', 'GDS_R2', 'GDS_R3', 'GDS_R4', 'GDS_R5']

# Implementación desde cero de Naive Bayes
class NaiveBayes:
    def fit(self, X, y):
        self.classes = set(y)
        self.class_counts = defaultdict(int)
        self.feature_counts = defaultdict(lambda: defaultdict(int))
        self.total = len(y)

        for xi, label in zip(X, y):
            self.class_counts[label] += 1
            for i, val in enumerate(xi):
                self.feature_counts[label][(i, val)] += 1

    def predict_one(self, xi):
        scores = {}
        for c in self.classes:
            logp = math.log(self.class_counts[c] / self.total)
            for i, val in enumerate(xi):
                freq = self.feature_counts[c][(i, val)] + 1
                total = self.class_counts[c] + 2
                logp += math.log(freq / total)
            scores[c] = logp
        return max(scores, key=scores.get)

# Función general de entrenamiento, predicción y evaluación con LOOCV
def evaluate_gds_column(df, gds_col, k_best=10):
    # Preparar datos
    X = df.drop(columns=['ID'] + gds_columns, errors='ignore')
    y = df[gds_col]

    # Aplicar selección de características chi2
    selector = SelectKBest(score_func=chi2, k=k_best)
    X_sel = selector.fit_transform(X, y)
    print(len(X_sel))

    y_true, y_pred = [], []
    for i in range(len(X_sel)):
        X_train = np.delete(X_sel, i, axis=0)
        y_train = np.delete(y, i)
        x_test = X_sel[i]
        y_test = y[i]

        model = NaiveBayes()
        model.fit(X_train, y_train)
        pred = model.predict_one(x_test)
        y_true.append(y_test)
        y_pred.append(pred)

    # Métricas macro promedio
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average='macro', zero_division=0),
        "recall": recall_score(y_true, y_pred, average='macro', zero_division=0),
        "f1_score": f1_score(y_true, y_pred, average='macro', zero_division=0)
    }

# Evaluar todas las agrupaciones
results = {}
for col in gds_columns:
    print(f"Evaluando codificación: {col}")
    results[col] = evaluate_gds_column(df, col, k_best=10)

# Convertir a DataFrame para mostrar
results_df = pd.DataFrame(results).T.round(4)
results_df.index.name = 'Codificación'
print("\nResultados por codificación de GDS:")
print(results_df)